In [40]:
import torch 
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from nltk import word_tokenize, sent_tokenize
import sys
import os
from pathlib import Path
import tiktoken
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
import json 
project_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(project_root))
from Components.MultiBlockDecoder import CompleteDecoderBlock

In [41]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()
trainer = BpeTrainer(
    vocab_size=8000, 
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
)

with open(r'../tiny shakespeare.txt', 'r') as file :
    text = file.read()
text

files = [r'../tiny shakespeare.txt']
tokenizer.train(files, trainer)
tokenizer.save("tokenizer-8k.json")
tokenised_corpus = tokenizer.encode(text)

In [11]:
with open("tokenizer-8k.json", "r", encoding="utf-8") as file:
    tokenizer_data = json.load(file)
vocab_dict = tokenizer_data.get("model",{}).get("vocab",{})
len(vocab_dict)

8000

In [12]:
class DatasetShakespiere(Dataset):
    def __init__(self, chunk_length, token_corpus):
        self.token_corpus = torch.tensor(token_corpus, dtype=torch.long)
        self.chunk_length = chunk_length
    def __len__(self):
        return (len(self.token_corpus)-1)//self.chunk_length
    
    def __getitem__(self, index):
        start = index * self.chunk_length
        end = start + self.chunk_length + 1
        
        window = self.token_corpus[start : end ]
        input = window[:-1]
        target = window[1:]
        
        return input, target

dataset = DatasetShakespiere(128, tokenised_corpus.ids)
data_loader = DataLoader( dataset, batch_size=128, pin_memory=True)

In [13]:
n_blocks = 2 
num_heads = 2
vocab_count = len(vocab_dict)
embed_dim = 512
ffo_neurons = 1024
model = CompleteDecoderBlock(n_blocks, num_heads, vocab_count, embed_dim, ffo_neurons, False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

CompleteDecoderBlock(
  (emebedding_layer): Embedding(8000, 512)
  (positional_layer): PositionalEncoder()
  (decoders): ModuleList(
    (0-1): 2 x SingleDecoderBlock(
      (ffo): Sequential(
        (0): Linear(in_features=512, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=512, bias=True)
      )
      (residual_attention): ResidualConnect(
        (sublayer): MultiHeadedAttention(
          (q): Linear(in_features=512, out_features=512, bias=True)
          (k): Linear(in_features=512, out_features=512, bias=True)
          (v): Linear(in_features=512, out_features=512, bias=True)
          (wo): Linear(in_features=512, out_features=512, bias=True)
        )
      )
      (residual_ffo): ResidualConnect(
        (sublayer): Sequential(
          (0): Linear(in_features=512, out_features=1024, bias=True)
          (1): ReLU()
          (2): Linear(in_features=1024, out_features=512, bias=True)
        )
      )
      (attention_n

In [14]:
a,b = next(iter(data_loader))
print(a.shape)
len(a.tolist())
a = a.to(device)

torch.Size([128, 128])


In [15]:
Criterion = nn.CrossEntropyLoss()
optimiser = optim.AdamW(model.parameters(), lr = 3e-4)

In [17]:
total_trained_epochs = 100

In [18]:
epochs = 200
model.train()
for epoch in range(epochs):
    epoch_loss = 0
    for input,target in data_loader:
            input = input.to(device, non_blocking = True)
            target = target.to(device, non_blocking = True)
            optimiser.zero_grad(set_to_none = True)
            
            with torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
                enabled=device.type == "cuda",
            ):
                logits = model(input) #logits = (B,T,V)
                logits = logits.flatten(0,1)
                target = target.flatten()
                loss = Criterion(logits, target)
                
            loss.backward()
            optimiser.step()
            epoch_loss += loss.detach()
            
            
    avg_loss = epoch_loss/len(data_loader)
    print(f"EPOCH : {epoch} , Avg Loss = {avg_loss}")
    total_trained_epochs +=1
            

EPOCH : 0 , Avg Loss = 0.5536541938781738
EPOCH : 1 , Avg Loss = 0.5343278050422668
EPOCH : 2 , Avg Loss = 0.5155104994773865
EPOCH : 3 , Avg Loss = 0.5016509890556335
EPOCH : 4 , Avg Loss = 0.4784126281738281
EPOCH : 5 , Avg Loss = 0.4555644690990448
EPOCH : 6 , Avg Loss = 0.43761157989501953
EPOCH : 7 , Avg Loss = 0.41958585381507874
EPOCH : 8 , Avg Loss = 0.4022344648838043
EPOCH : 9 , Avg Loss = 0.3816557228565216
EPOCH : 10 , Avg Loss = 0.36549538373947144
EPOCH : 11 , Avg Loss = 0.35488101840019226
EPOCH : 12 , Avg Loss = 0.34080609679222107
EPOCH : 13 , Avg Loss = 0.3331802487373352
EPOCH : 14 , Avg Loss = 0.3218652904033661
EPOCH : 15 , Avg Loss = 0.3119102418422699
EPOCH : 16 , Avg Loss = 0.29798054695129395
EPOCH : 17 , Avg Loss = 0.2882167398929596
EPOCH : 18 , Avg Loss = 0.2755606472492218
EPOCH : 19 , Avg Loss = 0.26504024863243103
EPOCH : 20 , Avg Loss = 0.2519885003566742
EPOCH : 21 , Avg Loss = 0.2412438690662384
EPOCH : 22 , Avg Loss = 0.23227517306804657
EPOCH : 23 , 

In [45]:
def inference_mode(text, max_len_generation):
    model.eval()
    arr_encoded = tokenizer.encode(text).ids
    
    for timestep in range(max_len_generation):
        input = torch.tensor([arr_encoded[-128:]], dtype=torch.long, device=device)
        # print(input.shape)
        output = model(input, return_post_softmax = False) # (1,T,V)
        vals,indices = torch.max(output[:,-1,:], dim = -1)
        arr_encoded.append(indices.tolist()[0])
        print(tokenizer.decode(arr_encoded))
inference_mode("This Took A turn for the wrong ",30)
    

This Took A turn for the wrong  for
This Took A turn for the wrong  for your
This Took A turn for the wrong  for your app
This Took A turn for the wrong  for your apple
This Took A turn for the wrong  for your apple.
This Took A turn for the wrong  for your apple.

This Took A turn for the wrong  for your apple.
So
This Took A turn for the wrong  for your apple.
So,
This Took A turn for the wrong  for your apple.
So, this
This Took A turn for the wrong  for your apple.
So, this,
This Took A turn for the wrong  for your apple.
So, this, unto
This Took A turn for the wrong  for your apple.
So, this, unto your
This Took A turn for the wrong  for your apple.
So, this, unto your to
This Took A turn for the wrong  for your apple.
So, this, unto your to your
This Took A turn for the wrong  for your apple.
So, this, unto your to your to
This Took A turn for the wrong  for your apple.
So, this, unto your to your to's
This Took A turn for the wrong  for your apple.
So, this, unto your to your to

In [46]:
checkpoint = {
    # Model state
    "model_state_dict": model.state_dict(),

    # Optimizer state, including AdamW momentum statistics
    "optimizer_state_dict": optimiser.state_dict(),

    # Training progress
    "epoch": total_trained_epochs,

    # Latest metrics
    "loss": avg_loss.item(),

    # Model configuration
    "model_config": {
        "n_blocks": n_blocks,
        "num_heads": num_heads,
        "vocab_count": vocab_count,
        "embed_dim": embed_dim,
        "ffo_neurons": ffo_neurons,
        "is_cross_attention": False,
        "context_length": 128,
    },

    # Training configuration
    "training_config": {
        "learning_rate": 3e-4,
        "batch_size": 128,
    },
}
torch.save(
    checkpoint,
    "checkpoint1.pt"
)